# Extended LSTM (xLSTM) for ADS-B Trajectory Prediction

This notebook demonstrates using Extended LSTM for predicting aircraft trajectories from ADS-B data.

**Key Features:**
- Exponential gating for better gradient flow
- Matrix memory (mLSTM) for richer context
- Enhanced long-range dependency modeling

**Use Cases:**
- Trajectory forecasting
- Collision avoidance
- Anomaly detection (via prediction deviation)
- Air traffic management

## 1. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add models to path
sys.path.append('..')

from xlstm.extended_lstm import create_xlstm_model, xLSTMTrajectoryPredictor
from data_utils import ADSBDataset

# Set style
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Load and Explore Data

In [ ]:
# Load ADS-B data
data_path = '../../golden_7day_eda_results/golden_7day_ml_dataset.csv'

if Path(data_path).exists():
    df = pd.read_csv(data_path)
    print(f"Loaded dataset: {len(df):,} records")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nFirst few rows:")
    display(df.head())
else:
    print(f"⚠️  Dataset not found at {data_path}")
    print("Please run the EDA script first to generate the ML dataset.")
    
    # Create dummy data for demonstration
    print("\nGenerating dummy data for demonstration...")
    n_samples = 5000
    df = pd.DataFrame({
        'timestamp': pd.date_range('2026-01-16', periods=n_samples, freq='1s'),
        'hex': np.random.choice(['ABC123', 'DEF456', 'GHI789'], n_samples),
        'lat': np.cumsum(np.random.randn(n_samples) * 0.001) + 60.0,
        'lon': np.cumsum(np.random.randn(n_samples) * 0.001) + 25.0,
        'alt_baro': np.cumsum(np.random.randn(n_samples) * 10) + 30000,
        'gs': np.random.randn(n_samples) * 10 + 450,
        'track': np.cumsum(np.random.randn(n_samples) * 2) % 360,
        'rssi': np.random.randn(n_samples) * 5 - 30,
    })

## 3. Create Dataset and DataLoader

In [ ]:
from torch.utils.data import DataLoader

# Select trajectory features
traj_features = ['lat', 'lon', 'alt_baro', 'gs', 'track', 'rssi']
available_features = [f for f in traj_features if f in df.columns]

# Create trajectory prediction dataset
dataset = ADSBDataset(
    df,
    sequence_length=20,
    prediction_horizon=10,
    features=available_features,
    normalize=True
)

print(f"Dataset created: {len(dataset)} sequences")
print(f"Features: {dataset.features}")
print(f"Input size: {len(dataset.features)}")

# Create data loader
data_loader = DataLoader(dataset, batch_size=32, shuffle=False)

# Get a sample
x_sample, y_sample = next(iter(data_loader))
print(f"\nSample batch:")
print(f"  Input shape: {x_sample.shape}")
print(f"  Target shape: {y_sample.shape}")

## 4. Create xLSTM Model

In [ ]:
# Model parameters
input_size = len(dataset.features)
hidden_size = 32
predict_steps = 10

# Create model
model = create_xlstm_model(
    input_size=input_size,
    model_type='trajectory_predictor',
    hidden_size=hidden_size,
    num_layers=2,
    predict_steps=predict_steps
).to(device)

print("xLSTM Trajectory Predictor created:")
print(f"  Input size: {input_size}")
print(f"  Hidden size: {hidden_size}")
print(f"  Prediction horizon: {predict_steps} steps")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Test Forward Pass

In [ ]:
# Test model on sample data
model.eval()
with torch.no_grad():
    x_test = x_sample[:4].to(device)  # Take 4 samples
    
    # Next-step prediction
    next_step = model(x_test, predict_future=False)
    print("Next-step prediction:")
    print(f"  Input shape: {x_test.shape}")
    print(f"  Prediction shape: {next_step.shape}")
    
    # Future trajectory prediction
    future = model(x_test, predict_future=True)
    print(f"\nFuture trajectory prediction:")
    print(f"  Input shape: {x_test.shape}")
    print(f"  Future prediction shape: {future.shape}")
    print(f"\nPredicted {predict_steps} steps into the future!")

## 6. Visualize Predictions

In [ ]:
# Generate predictions for visualization
x_vis = x_test.cpu()
y_vis = y_sample[:4].cpu()
future_vis = future.cpu()

# Plot trajectories
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for i in range(4):
    ax = axes[i // 2, i % 2]
    
    # Historical trajectory
    hist_steps = range(x_vis.shape[1])
    ax.plot(hist_steps, x_vis[i, :, 0], 'b-o', label='Historical', linewidth=2, markersize=4)
    
    # Future ground truth
    future_steps = range(x_vis.shape[1], x_vis.shape[1] + y_vis.shape[1])
    ax.plot(future_steps, y_vis[i, :, 0], 'g-o', label='Ground Truth', linewidth=2, markersize=4)
    
    # Predicted future
    ax.plot(future_steps, future_vis[i, :, 0], 'r--s', label='Prediction', linewidth=2, markersize=4)
    
    # Add vertical line separating history and future
    ax.axvline(x=x_vis.shape[1]-0.5, color='gray', linestyle=':', linewidth=2, alpha=0.5)
    ax.text(x_vis.shape[1]-0.5, ax.get_ylim()[0], ' Present', 
           verticalalignment='bottom', fontsize=10, color='gray')
    
    ax.set_xlabel('Time Step', fontsize=11)
    ax.set_ylabel('Feature Value (normalized)', fontsize=11)
    ax.set_title(f'Sample {i+1}: Trajectory Prediction', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compute prediction error
pred_error = torch.mean(torch.abs(future_vis - y_vis)).item()
print(f"\nMean Absolute Error: {pred_error:.4f}")

## 7. Compare with Ground Truth (Multi-Feature)

In [ ]:
# Plot all features for one sample
sample_idx = 0
feature_names = dataset.features[:min(4, len(dataset.features))]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for feat_idx, feat_name in enumerate(feature_names):
    ax = axes[feat_idx]
    
    # Historical
    hist_steps = range(x_vis.shape[1])
    ax.plot(hist_steps, x_vis[sample_idx, :, feat_idx], 'b-o', 
           label='Historical', linewidth=2, markersize=3, alpha=0.7)
    
    # Future
    future_steps = range(x_vis.shape[1], x_vis.shape[1] + y_vis.shape[1])
    ax.plot(future_steps, y_vis[sample_idx, :, feat_idx], 'g-o', 
           label='Ground Truth', linewidth=2, markersize=4)
    ax.plot(future_steps, future_vis[sample_idx, :, feat_idx], 'r--s', 
           label='Prediction', linewidth=2, markersize=4)
    
    ax.axvline(x=x_vis.shape[1]-0.5, color='gray', linestyle=':', linewidth=2, alpha=0.3)
    
    ax.set_xlabel('Time Step', fontsize=11)
    ax.set_ylabel('Normalized Value', fontsize=11)
    ax.set_title(f'Feature: {feat_name}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Multi-Feature Trajectory Prediction', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 8. Training (Quick Demo)

In [ ]:
# Quick training demo (5 epochs)
model.train()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

num_epochs = 5
losses = []

print("Training for 5 epochs (demo)...")
for epoch in range(num_epochs):
    epoch_loss = 0
    for x, y in data_loader:
        x = x.to(device)
        y = y.to(device)
        
        optimizer.zero_grad()
        
        # Next-step prediction
        predictions = model(x, predict_future=False)
        loss = criterion(predictions[:, :-1, :], x[:, 1:, :])
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(data_loader)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{num_epochs}: Loss = {avg_loss:.6f}")

# Plot training curve
plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs+1), losses, 'b-o', linewidth=2, markersize=8)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss (MSE)', fontsize=12)
plt.title('xLSTM Training Progress (Demo)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

print("\n✅ Training demo complete!")

## 9. Summary and Next Steps

### What We've Done:
1. ✅ Loaded ADS-B trajectory data
2. ✅ Built xLSTM trajectory predictor with exponential gating
3. ✅ Tested next-step and multi-step predictions
4. ✅ Visualized predictions vs ground truth
5. ✅ Demonstrated quick training loop

### For Full Training:

```bash
python analysis/models/xlstm/train_xlstm_trajectory.py \
    --data_path analysis/golden_7day_eda_results/golden_7day_ml_dataset.csv \
    --epochs 100 \
    --hidden_size 64 \
    --prediction_horizon 20
```

### Applications:
- **Trajectory Forecasting**: Predict future aircraft positions
- **Collision Avoidance**: Early warning systems
- **Anomaly Detection**: Detect deviations from predicted paths
- **Air Traffic Management**: Optimize routes and spacing
- **Spoofing Detection**: Identify impossible trajectory changes

### Key Advantages:
- ⚡ **Exponential Gating**: Better gradient flow for long sequences
- 🧠 **Matrix Memory**: Richer context representation
- 📈 **Scalable**: Handles long sequences efficiently
- 🎯 **Accurate**: Improved prediction over standard LSTM
- 🔄 **Flexible**: Both single-step and multi-step prediction

### Model Components:
- **sLSTM**: Scalar memory with exponential gates
- **mLSTM**: Matrix memory with covariance updates
- **Residual Connections**: Improved training stability
- **Layer Normalization**: Better convergence